# IEEE-CIS Fraud Detection — when naive CV leaks, group/time CV tells the truth

Case: [IEEE-CIS Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection)
(Vesta e-commerce transactions; **≈3% fraud** in the sampled window, ~430
features after a 1:1 join of the transaction and identity tables, a
forward-in-time public/private split).

This is the showcase the first four notebooks could not be — they are all i.i.d.
tabular. Fraud data has **two** structures that an i.i.d. CV quietly violates:

- **repeated cards**: the same card recurs across many transactions, so a random
  fold puts a card in both train and test — the model memorizes the card, not
  fraud. The engineered `uid = card1 + addr1 + D1` exposes that entity for
  **group-aware CV** (`groups=`), where no card spans the split;
- **time**: `TransactionDT` orders the data and the leaderboard split is forward
  in time, so an honest estimate needs **time-series CV** with purge/embargo
  (`time=`), not a shuffled fold that trains on the future.

The metric is **`roc_auc`** — the exact competition metric (1:1, native). The
story below: a naive i.i.d. run looks great and is *optimistic*; the group- and
time-aware runs reveal the honest number.

> Data note: downloaded from a community Kaggle **mirror**
> (`lnasiri007/ieeecis-fraud-detection`); the files are identical to the
> competition. `SAMPLE_ROWS` keeps a contiguous early-time window so the runs
> finish in minutes, not hours — set it to `None` for the full 590k rows. Live
> submission stays off.

In [1]:
import json
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd

from honestml import (
    AutoML,
    CVConfig,
    FeatureSelectionConfig,
    HPOConfig,
    render_report,
    save_run_report,
)

SEED = 42
SAMPLE_ROWS = 200_000  # contiguous early-time window; set to None for the full 590k rows
DATA = Path("data/ieee-fraud")
RESULTS = Path("results/ieee-fraud")
RESULTS.mkdir(parents=True, exist_ok=True)
KAGGLE = os.environ.get("KAGGLE_BIN") or shutil.which("kaggle") or str(Path.home() / ".local" / "bin" / "kaggle")

logging.basicConfig(format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("honestml").setLevel(logging.INFO)

In [2]:
if not (DATA / "train_transaction.csv").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [KAGGLE, "datasets", "download", "-d", "lnasiri007/ieeecis-fraud-detection", "-p", str(DATA), "--unzip"],
        check=True,
    )

In [3]:
tx = pd.read_csv(DATA / "train_transaction.csv")
idf = pd.read_csv(DATA / "train_identity.csv")
df = tx.merge(idf, on="TransactionID", how="left").sort_values("TransactionDT").reset_index(drop=True)
if SAMPLE_ROWS is not None:
    df = df.iloc[:SAMPLE_ROWS].copy()

# the engineered client entity (the well-known 'magic uid'): same card recurs across transactions
uid = (
    df["card1"].astype("string").fillna("NA")
    + "_" + df["addr1"].astype("string").fillna("NA")
    + "_" + df["D1"].astype("string").fillna("NA")
).to_numpy()
tdt = df["TransactionDT"].to_numpy()  # the time axis (seconds from a fixed reference)
y = df.pop("isFraud")
X = df.drop(columns=["TransactionID", "TransactionDT"])  # ids/time axis are metadata, not features

n_groups = len(set(uid.tolist()))
rep = len(uid) / n_groups
print(f"rows: {len(X)}, features: {X.shape[1]}, fraud: {y.mean():.3%}")
print(f"uid groups: {n_groups} ({rep:.1f} transactions per card on average)")
X.head()

rows: 200000, features: 431, fraud: 3.012%
uid groups: 91869 (2.2 transactions per card on average)


,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,68.5,W,13926,NaN,150.0,discover,142.0,credit,315.0,87.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,325.0,87.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,59.0,W,4663,490.0,150.0,visa,166.0,debit,330.0,87.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,476.0,87.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,420.0,87.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


## Run A — naive i.i.d. CV (the optimistic baseline)

A plain 5-fold **stratified** CV + a stratified 20% holdout. This ignores both
structures: a card can land in train and test, and a fold can train on later
transactions than it scores. It is exactly what most quick baselines do — and,
as the honest contour will show, it flatters the model.

In [4]:
model_iid = AutoML(
    task="binary",
    metric="roc_auc",
    cv=CVConfig(outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model_iid.fit(X, y)
print(f"naive i.i.d. fit took {(time.perf_counter() - t0) / 60:.1f} min")
rep_iid = model_iid.run_report_
sel_iid = next(e["score"] for e in rep_iid["leaderboard"] if e["model_id"] == rep_iid["winner"])
print(f"winner {rep_iid['winner']}  selection roc_auc {sel_iid:.4f}  holdout {rep_iid['holdout_score']:.4f}  optimism {sel_iid - rep_iid['holdout_score']:+.4f}")
pd.DataFrame(rep_iid["leaderboard"])

INFO honestml.adapters.reader: auto-typing column=V107 dtype=Float64 role=ignore reason=constant


INFO honestml.adapters.reader: auto-typing column=id_33 dtype=String role=categorical reason=high_cardinality_string


INFO honestml.adapters.reader: auto-typing column=DeviceInfo dtype=String role=categorical reason=high_cardinality_string


WARNING honestml: native categorical gate demoted 4 high-cardinality column(s) to ordinal codes: ['id_30', 'id_31', 'id_33', 'DeviceInfo'] (native_cat_max_unique=64)


INFO honestml: stage key=run stage=selection elapsed=1056.5s


...

INFO honestml: stage key=run stage=refit elapsed=6.4s


INFO honestml: stage key=run stage=refit elapsed=7.2s


INFO honestml: stage key=run stage=finalize elapsed=7.2s


naive i.i.d. fit took 17.9 min
winner lightgbm  selection roc_auc 0.9515  holdout 0.9501  optimism +0.0015


,model_id,score,rank
0,lightgbm,0.951535,1
1,xgboost,0.944984,2
2,catboost,0.932963,3
3,linear,0.857911,4
4,baseline,0.499914,5


## Run B — group-aware CV (anti-leakage across a card's transactions)

Same zoo, but `cv=CVConfig(scheme="group")` + `groups=uid`: every transaction of
a card stays on one side of the split, and the outer holdout is **group-disjoint**
too (finding #11). The OOF cross-fit target encoding and calibration become
leakage-free across the entity. If the i.i.d. run was memorizing cards, the
honest holdout here drops.

In [5]:
model_grp = AutoML(
    task="binary",
    metric="roc_auc",
    cv=CVConfig(scheme="group", outer_holdout=0.2, calibrate="auto"),
    random_state=SEED,
)
t0 = time.perf_counter()
model_grp.fit(X, y, groups=uid)
print(f"group-aware fit took {(time.perf_counter() - t0) / 60:.1f} min")
rep_grp = model_grp.run_report_
sel_grp = next(e["score"] for e in rep_grp["leaderboard"] if e["model_id"] == rep_grp["winner"])
print(f"winner {rep_grp['winner']}  selection roc_auc {sel_grp:.4f}  holdout {rep_grp['holdout_score']:.4f}  optimism {sel_grp - rep_grp['holdout_score']:+.4f}")
print(f"naive i.i.d. holdout was {rep_iid['holdout_score']:.4f}; group-honest holdout is {rep_grp['holdout_score']:.4f}")
pd.DataFrame(rep_grp["leaderboard"])

INFO honestml.adapters.reader: auto-typing column=V107 dtype=Float64 role=ignore reason=constant


INFO honestml.adapters.reader: auto-typing column=id_33 dtype=String role=categorical reason=high_cardinality_string


INFO honestml.adapters.reader: auto-typing column=DeviceInfo dtype=String role=categorical reason=high_cardinality_string


WARNING honestml: native categorical gate demoted 4 high-cardinality column(s) to ordinal codes: ['id_30', 'id_31', 'id_33', 'DeviceInfo'] (native_cat_max_unique=64)


INFO honestml: stage key=run stage=selection elapsed=1081.1s


INFO honestml: stage key=run stage=refit elapsed=6.1s


INFO honestml: stage key=run stage=refit elapsed=7.1s


INFO honestml: stage key=run stage=finalize elapsed=7.1s


group-aware fit took 18.3 min
winner lightgbm  selection roc_auc 0.9324  holdout 0.9494  optimism -0.0171
naive i.i.d. holdout was 0.9501; group-honest holdout is 0.9494


,model_id,score,rank
0,lightgbm,0.932350,1
1,xgboost,0.921929,2
2,catboost,0.919876,3
3,linear,0.847579,4
4,baseline,0.486898,5


## Run C — level 2: time-aware CV + feature selection + HPO (matches the leaderboard split)

The competition's public/private split is forward in time, so the honest
estimate uses **time-series CV** on `TransactionDT` with a wall-clock
purge/embargo, plus the level-2 machinery on the big noisy feature store:

- `cv=CVConfig(scheme="timeseries", ...)` + `time=TransactionDT` — an expanding
  window with a **wall-clock** purge/embargo (`purge_delta`/`embargo_delta`):
  `TransactionDT` is in seconds and transactions arrive irregularly, so a gap
  counted in rows spans anywhere from a few hours to nearly a day; a fixed
  **one-day** Δt gap pins the real serial-correlation window around each test block;
- `FeatureSelectionConfig(strategy="importance", cutoff="auto")` — the 339
  anonymized `V*` columns are highly redundant; how many survive, with the honest
  no-selection gate;
- `preset="best"` + `HPOConfig` — significance-gated ensembling and Optuna tuning;
- calibration and refinement selection are *disabled under time-series CV* (an
  expanding-window gate would look ahead) — the report says so, it is not silent.

In [6]:
n_test = max(2000, len(y) // 12)  # rows per time fold — large enough for both classes at 3.5% fraud
one_day = 24 * 3600  # wall-clock purge/embargo: a fixed real duration, since TransactionDT is in seconds
model_ts = AutoML(
    task="binary",
    metric="roc_auc",
    preset="best",
    hpo=HPOConfig(n_trials=15, timeout_s=600),
    feature_selection=FeatureSelectionConfig(strategy="importance", cutoff="auto"),
    cv=CVConfig(scheme="timeseries", n_splits=5, n_test=n_test, purge_delta=one_day, embargo_delta=one_day, outer_holdout=0.2),
    random_state=SEED,
)
t0 = time.perf_counter()
model_ts.fit(X, y, time=tdt)
print(f"time-aware level-2 fit took {(time.perf_counter() - t0) / 60:.1f} min")

INFO honestml.composition.presets: preset best applied: ['ensemble']


INFO honestml.adapters.reader: auto-typing column=V107 dtype=Float64 role=ignore reason=constant


INFO honestml.adapters.reader: auto-typing column=id_33 dtype=String role=categorical reason=high_cardinality_string


INFO honestml.adapters.reader: auto-typing column=DeviceInfo dtype=String role=categorical reason=high_cardinality_string


C:\Users\Admin\Documents\AutoML\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...

WARNING honestml.adapters.boosting: boosting 'xgboost' trained without early stopping; leaderboard comparison may favor overfit settings


INFO honestml: stage key=run stage=refit elapsed=52.7s


INFO honestml: stage key=run stage=refit elapsed=66.2s


INFO honestml: stage key=run stage=finalize elapsed=66.2s


time-aware level-2 fit took 42.4 min


In [7]:
rep_ts = model_ts.run_report_
sel_ts = next(e["score"] for e in rep_ts["leaderboard"] if e["model_id"] == rep_ts["winner"])
print(f"winner          : {rep_ts['winner']}")
print(f"selection score : {sel_ts:.4f}   (time-series OOF roc_auc)")
print(f"holdout score   : {rep_ts['holdout_score']:.4f}   (late-window holdout)")
print(f"optimism        : {sel_ts - rep_ts['holdout_score']:+.4f}")
print()
print("fs      :", json.dumps(rep_ts["feature_selection"], default=str)[:400])
print("hpo     :", json.dumps(rep_ts["hpo"], default=str)[:400])
print("ensemble:", json.dumps(rep_ts["ensemble"], default=str)[:300])
save_run_report(rep_ts, RESULTS)
md_path = render_report(rep_ts, RESULTS, fmt="md")
pd.DataFrame(rep_ts["leaderboard"])

winner          : lightgbm
selection score : 0.8563   (time-series OOF roc_auc)
holdout score   : 0.9071   (late-window holdout)
optimism        : -0.0508

fs      : {"strategy": "importance", "n_selected": 137, "selected": ["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "dist1", "C1", "C2", "C4", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C14", "D1", "D2", "D3", "D4", "D5", "D7", "D8", "D9", "D10", "D12", "D15", "V12", "V13", "V19", "V20", "V24", "V37", "V38", "V44", "V45", "V47", "V53", "V54", "V56", "V61", "V62", "V67", "V75", "V76
hpo     : {"backend": "optuna", "inner_cv": 3, "deterministic": false, "selection_oof_is_post_tuning": true, "tuned_on_full_feature_space": true, "cost_estimate_fits": 135, "tuned": {"catboost": {"chosen_params": {"depth": 6, "learning_rate": 0.2536999076681772, "iterations": 400, "l2_leaf_reg": 3.968793330444372, "subsample": 0.6624074561769746, "one_hot_max_size": 11}, "inner_best_score": 0.89259093076044
ensemble: {"applied": 

,model_id,score,rank
0,catboost,0.892212,1
1,xgboost,0.857437,2
2,lightgbm,0.856297,3
3,linear,0.819156,4
4,baseline,0.471704,5


## What the three runs say

The honest contour is the whole point, and it shows up in the **selection
scores** as the CV stops ignoring the data's structure:

| regime | selection OOF roc_auc | holdout roc_auc |
| --- | --- | --- |
| naive i.i.d. | 0.9515 | 0.9501 |
| group-aware (uid) | 0.9324 | 0.9494 |
| time-aware (level 2) | 0.8563 | 0.9071 |

**The naive i.i.d. number is optimistic.** A random fold lets a card sit in both
train and test, so the model scores partly by recognizing cards it has already
seen — worth ~2 AUC points: with no card spanning the split, the group-aware OOF
falls to 0.9324. The bigger correction is **time**. Scoring fraud *forward in
time*, the way the leaderboard actually does — now with a fixed **one-day
wall-clock** purge/embargo (`purge_delta`/`embargo_delta`) around each test block
rather than a row count that drifts with transaction density — drops the honest
OOF to **0.8563**, more than **9 points** below the i.i.d. estimate. The
late-window holdout (0.9071) lands above that forward OOF but still well under the
i.i.d. number: a model that looks like 0.95 on shuffled folds is really a
~0.86–0.91 model on next month's transactions. That gap is exactly what an i.i.d.
CV hides and a `time=`/`groups=` CV refuses to.

This is also where the **level-2 machinery earns its keep — honestly**, all under
the time-series CV:

- **feature selection** cut the store from **430 to 137** features: most of the
  339 anonymized `V*` columns are redundant, and `importance` / `cutoff=auto`
  pruned them without losing AUC (the no-selection gate let the cut through);
- **HPO** tuned each boosting on an inner CV;
- **Caruana ensembling cleared its significance gate** and **shipped a blend** of
  the candidates, dominated by **lightgbm (weight ~0.47)** with linear and
  catboost next — the honest contour lets ensembling improve the model only because
  the blend beats the single winner out-of-fold.

Read together: an i.i.d. CV would have shipped an over-confident model and a
leaderboard-chasing story; the group/time-aware contour reports the number you
would actually get in production — and still lets FS, HPO and ensembling improve
it without lying about it.